# Data Cleaning

## Objective

The goal of this notebook is to assess the quality of the IBM Telco Customer Churn dataset and prepare a clean version for preprocessing and machine learning. we will identify and resolve the data quality issues such as duplicate records, hidden missing valuse, incorrect data types, and invalid intries while preserveing the integrity of the original dataset.

In [19]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [20]:
df = pd.read_csv('../data/raw/Telco-Customer-churn.csv')

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# Data Quality Assessment

Before modifying the dataset, we will evaluate its overall quality. This assessment focuses on indentify duplicate records, missing values, hidden missing values, incorrect data types, and inconsistent data that may negatively affect downstream analysis and machine learning models.

## Duplicate Records

### Business Question

Dose the dataset contain duplicate customer records that could bias the analysis or model performance?

In [21]:
df.duplicated().sum()

np.int64(0)

### Key Findings

No duplicate records were found in the dataset. Each row represents a unique customer, indicating that duplicate observations are unlikly to interduce bias into the analysis and predictive model.

## Missing Values

### Business Question

Are there any explicite or hidden missing values that may require cleaning before analysis and modeling?

In [22]:
df.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [23]:
(df == "").sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [24]:
df['TotalCharges'].value_counts().head()

TotalCharges
20.2     11
         11
19.75     9
19.9      8
20.05     8
Name: count, dtype: int64

### Insight

NO explicit missing values are detected using `isna()`. However, hidden missing values are represented as blank strings are present in `TotalCharges` column. This values prevent pandas from recognizing the column as numeric and must be addressed before modeling.

## Data Types

### Business Question

Are all variables stored using appropriate data types?

In [25]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

In [26]:
df.select_dtypes(include='object').columns

C:\Users\rasooly com\AppData\Local\Temp\ipykernel_19064\3732952691.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include='object').columns


Index(['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges',
       'Churn'],
      dtype='str')

### Insight 

- **customerID:** customerID is identifier, this consists from uniqe numbers and letters that my not help in analysis and model preformence.

- **TotalCharge:** Due to blank strings valuse in column, pandas identify this column as categorical and we should convert this to numeric.

- **Churn:** is categorical target.

## Invalid Values 

### Business Question

Do any numeric variables contain impossible or unexpected values?

In [27]:
df[['tenure', 'MonthlyCharges']].describe()

,tenure,MonthlyCharges
count,7043.000000,7043.000000
mean,32.371149,64.761692
std,24.559481,30.090047
min,0.000000,18.250000
25%,9.000000,35.500000
50%,29.000000,70.350000
75%,55.000000,89.850000
max,72.000000,118.750000


In [28]:
df['tenure'].min(), df['tenure'].max()

(np.int64(0), np.int64(72))

In [29]:
df['MonthlyCharges'].min(), df['MonthlyCharges'].max()

(np.float64(18.25), np.float64(118.75))

### Insight

No invalid numerical valuse were detected in the inspected variables. The observation range are consistent with business context of customer subscriptions.

# Cleaning

In [30]:
# standardize the columns name by converting to lowercase, stripping whitespaces, and snake_case.

df.columns = df.columns.str.lower().str.strip()

# rename all multi-word columns to snake_case and saving the result.
df.rename(columns={'customerid':'customer_id',
                   'seniorcitizen':'senior_citizen', 
                   'phoneservice':'phone_service',
                   'multiplelines':'multiple_lines', 
                   'internetservice':'internet_service', 
                   'onlinesecurity':'online_security',
                   'onlinebackup':'online_backup',
                   'deviceprotection':'device_protection',
                   'techsupport':'tech_support',
                   'streamingtv':'streaming_tv',
                   'streamingmovies':'streaming_movies',
                   'paperlessbilling':'paperless_billing',
                   'paymentmethod':'payment_method',
                   'monthlycharges':'monthly_charges',
                   'totalcharges':'total_charges'
                   }, inplace=True)

In [31]:
# replace spaces with NaN and then convert to float
df['total_charges'] = df['total_charges'].replace(' ',np.nan).astype(float)

In [32]:
# change the NaN values to zero 

df['total_charges'] = df['total_charges'].fillna(0)

-  The missing values in ``total_charges`` belong to customers with **zero** tenure who have not yet accumulated any charges. Therefore, replacing these missing values with **0** preserves the business meaning of the data.

In [33]:
df.head()

,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [34]:
# final check the if the changes applied
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        7043 non-null   str    
 1   gender             7043 non-null   str    
 2   senior_citizen     7043 non-null   int64  
 3   partner            7043 non-null   str    
 4   dependents         7043 non-null   str    
 5   tenure             7043 non-null   int64  
 6   phone_service      7043 non-null   str    
 7   multiple_lines     7043 non-null   str    
 8   internet_service   7043 non-null   str    
 9   online_security    7043 non-null   str    
 10  online_backup      7043 non-null   str    
 11  device_protection  7043 non-null   str    
 12  tech_support       7043 non-null   str    
 13  streaming_tv       7043 non-null   str    
 14  streaming_movies   7043 non-null   str    
 15  contract           7043 non-null   str    
 16  paperless_billing  7043 non-null   

In [35]:
df.isna().sum()

customer_id          0
gender               0
senior_citizen       0
partner              0
dependents           0
tenure               0
phone_service        0
multiple_lines       0
internet_service     0
online_security      0
online_backup        0
device_protection    0
tech_support         0
streaming_tv         0
streaming_movies     0
contract             0
paperless_billing    0
payment_method       0
monthly_charges      0
total_charges        0
churn                0
dtype: int64

In [36]:
df.describe()

,senior_citizen,tenure,monthly_charges,total_charges
count,7043.000000,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692,2279.734304
std,0.368612,24.559481,30.090047,2266.794470
min,0.000000,0.000000,18.250000,0.000000
25%,0.000000,9.000000,35.500000,398.550000
50%,0.000000,29.000000,70.350000,1394.550000
75%,0.000000,55.000000,89.850000,3786.600000
max,1.000000,72.000000,118.750000,8684.800000


# Final Summary

### Problems that we found:
- The `total_charges` column had some hidden missing values by blank srings and these values prevent pandas to recognize the column as numeric and the `total_charges` column detected as string.

### Changes that we made:
- For better readability we renamed the columns name to **lowercase** and **snake_case**.

- The *missing values* in `total_charges` first changed to **NaN** with `np.nan` method, then filled with **Zero (0)**.

- The *data type* of `total_charges` column convert to **float**.

### Dataset ready for preprocessing:
- Now the dataset is ready for preprocessing and machine learning.

In [37]:
# save the clean dataset
df.to_csv(
    "../data/processed/customer_churn_clean.csv",
    index=False
)

The cleaned dataset has been saved to the processed directory and will be used in subsequent notebooks.